In [14]:
import os 
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_deepseek import ChatDeepSeek
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [15]:
load_dotenv()

True

In [7]:
docs = [
    Document(
        page_content=(
            "Artificial intelligence has made remarkable strides in natural language processing, "
            "with large language models now capable of generating human-quality text and code. "
            "Computer vision systems can identify objects in images with superhuman accuracy, "
            "powering applications from autonomous vehicles to medical imaging diagnostics. "
            "However, the rapid advancement of AI has raised significant ethical concerns about "
            "job displacement, algorithmic bias, and the concentration of power among a few tech companies."
        ),
        metadata={"topic": "artificial_intelligence"},
    ),
    Document(
        page_content=(
            "Global temperatures have risen by approximately 1.1 degrees Celsius since pre-industrial "
            "times, driven primarily by the burning of fossil fuels. The melting of polar ice caps has "
            "accelerated, contributing to rising sea levels that threaten coastal communities worldwide. "
            "Renewable energy adoption is growing rapidly, with solar and wind power becoming cheaper "
            "than coal in many regions. Governments are implementing carbon pricing mechanisms and "
            "investing in green infrastructure to meet Paris Agreement targets."
        ),
        metadata={"topic": "climate_change"},
    ),
    Document(
        page_content=(
            "NASA's Artemis program aims to return humans to the Moon by the mid-2020s, establishing "
            "a sustainable presence as a stepping stone to Mars. Private companies like SpaceX are "
            "developing reusable rocket technology that has dramatically reduced launch costs. "
            "The James Webb Space Telescope has captured unprecedented images of distant galaxies, "
            "revealing new insights about the early universe. Asteroid mining is being explored as a "
            "potential source of rare minerals needed for electronics manufacturing."
        ),
        metadata={"topic": "space_exploration"},
    ),
    Document(
        page_content=(
            "CRISPR gene editing technology has revolutionized medical genomics, enabling precise "
            "modifications to DNA sequences that were previously impossible. Researchers are using "
            "genomic data to develop personalized medicine approaches, tailoring treatments based on "
            "an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by "
            "COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare "
            "genetic disorders. Hospital information systems are increasingly integrating genomic data "
            "to support clinical decision-making at the point of care."
        ),
        metadata={"topic": "medicine"},
    ),
    Document(
        page_content=(
            "The global economy is navigating a period of high inflation driven by supply chain "
            "disruptions, energy price volatility, and post-pandemic demand surges. Central banks "
            "worldwide have raised interest rates aggressively to combat inflation, impacting housing "
            "markets and consumer spending. Cryptocurrency regulation is becoming a priority for "
            "financial authorities, with the EU's MiCA framework setting a global precedent. "
            "Trade tensions between major economies continue to reshape global supply chains, "
            "pushing companies toward nearshoring and diversification strategies."
        ),
        metadata={"topic": "economics"},
    ),
    Document(
        page_content=(
            "Quantum computing has reached a critical milestone with several companies demonstrating "
            "quantum advantage on specific computational tasks. Error correction remains the biggest "
            "challenge, as current quantum processors are highly susceptible to noise and decoherence. "
            "Quantum simulation of molecular structures could transform drug discovery by accurately "
            "modeling protein folding and chemical interactions. Major tech companies and governments "
            "are investing billions in quantum research, viewing it as essential for national security "
            "and economic competitiveness."
        ),
        metadata={"topic": "quantum_computing"},
    ),
]

In [9]:
##split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(docs)

In [10]:
##Building in-memory chroma vector store
vector_store = Chroma.from_documents(splits, embedding=OllamaEmbeddings(model="embeddinggemma:latest"),
    
                                    collection_name = "re-ranking-collection")

In [11]:
## creating a base retriever from the vector store
base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [12]:
query = "What are the latest advancements in AI and their ethical implications?"
base_retriever_results = base_retriever.invoke(query)
for i,base_retriever_result in enumerate(base_retriever_results):
    print(f"Base Retriever Result {i+1}: {base_retriever_result.page_content}\n")

Base Retriever Result 1: Artificial intelligence has made remarkable strides in natural language processing, with large language models now capable of generating human-quality text and code. Computer vision systems can identify objects in images with superhuman accuracy, powering applications from autonomous vehicles to medical imaging diagnostics. However, the rapid advancement of AI has raised significant ethical concerns about job displacement, algorithmic bias, and the concentration of power among a few tech

Base Retriever Result 2: concerns about job displacement, algorithmic bias, and the concentration of power among a few tech companies.

Base Retriever Result 3: The global economy is navigating a period of high inflation driven by supply chain disruptions, energy price volatility, and post-pandemic demand surges. Central banks worldwide have raised interest rates aggressively to combat inflation, impacting housing markets and consumer spending. Cryptocurrency regulation is bec

In [17]:
compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2")
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=base_retriever)
reranked_results = compression_retriever.invoke(query)
for i,reranked_result in enumerate(reranked_results):
    print(f"Reranked Result {i+1}: {reranked_result.page_content}\n")

PydanticUserError: `FlashrankRerank` is not fully defined; you should define `Ranker`, then call `FlashrankRerank.model_rebuild()`.

For further information visit https://errors.pydantic.dev/2.13/u/class-not-fully-defined

In [18]:
compressor = FlashrankRerank()  # remove the stray quote/extra arg; pass model if your implementation requires a model param
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=base_retriever)
reranked_results = compression_retriever.get_relevant_documents(query)
for i, reranked_result in enumerate(reranked_results):
    print(f"Reranked Result {i+1}: {reranked_result.page_content}\n")

PydanticUserError: `FlashrankRerank` is not fully defined; you should define `Ranker`, then call `FlashrankRerank.model_rebuild()`.

For further information visit https://errors.pydantic.dev/2.13/u/class-not-fully-defined